# Fuente 1 - API de Siniestralidad de Medellín

## Objetivo

Consultar la API de Siniestralidad de la Secretaría de Movilidad de Medellín para conocer su estructura, identificar las variables disponibles y verificar si contiene la información temporal, descriptiva y geográfica necesaria para el proyecto sobre la curva de La Asomadera #2 en la Avenida Las Palmas.

En esta etapa el propósito es conocer la fuente, comprobar que se puede acceder a ella e identificar las variables que posteriormente permitirán relacionarla con las demás fuentes del proyecto.

## 1. Identificación de la fuente

**Nombre:** Siniestralidad  
**Entidad:** Secretaría de Movilidad de Medellín - Observatorio de Movilidad  
**Tipo de fuente:** API REST de ArcGIS  
**Información disponible:** Registros georreferenciados (para los años 2019 a 2025) de siniestros viales ocurridos en Medellín.  
**Periodo informado por la fuente:** 2019 a 2025.

Primero se consultarán los metadatos de la API para conocer su estructura antes de descargar registros y comprobar conexion a nuestra fuente.

fuente: https://www.medellin.gov.co/servidormapas/rest/services/transporte/VM_Accidentes/MapServer/0


In [ ]:
import requests
import pandas as pd

# URL de la capa de Siniestralidad de la Secretaría de Movilidad de Medellín
url_capa = "https://www.medellin.gov.co/servidormapas/rest/services/transporte/VM_Accidentes/MapServer/0"

# Consultamos los metadatos de la capa en formato JSON
respuesta = requests.get(
    url_capa,
    params={"f": "json"},
    timeout=30
)

respuesta.raise_for_status()

metadata = respuesta.json()

print("Conexión realizada correctamente")
print("Nombre de la capa:", metadata.get("name"))
print("Tipo de capa:", metadata.get("type"))
print("Tipo de geometría:", metadata.get("geometryType"))
print("Cantidad máxima de registros por consulta:", metadata.get("maxRecordCount"))

Conexión realizada correctamente
Nombre de la capa: Siniestralidad
Tipo de capa: Feature Layer
Tipo de geometría: esriGeometryPoint
Cantidad máxima de registros por consulta: 2000


## 2. Variables disponibles en la fuente

A partir de los metadatos de la API se consultan los campos disponibles. Esto permite conocer las variables reales de la fuente antes de seleccionar cuáles serán utilizadas en el proyecto.

In [ ]:
campos = metadata.get("fields", [])

tabla_campos = pd.DataFrame([
    {
        "nombre_variable": campo.get("name"),
        "descripcion": campo.get("alias"),
        "tipo_arcgis": campo.get("type")
    }
    for campo in campos
])

tabla_campos

,nombre_variable,descripcion,tipo_arcgis
0,objectid,OBJECTID,esriFieldTypeOID
1,radicado,radicado,esriFieldTypeDouble
2,fecha_siniestro,fecha_siniestro,esriFieldTypeString
3,hora_siniestro,hora_siniestro,esriFieldTypeString
4,dia,dia,esriFieldTypeString
5,mes,mes,esriFieldTypeString
6,anio,anio,esriFieldTypeDouble
7,clase_siniestro,clase_siniestro,esriFieldTypeString
8,gravedad_siniestro,gravedad_siniestro,esriFieldTypeString
9,direccion,direccion,esriFieldTypeString


### Interpretación

La consulta de metadatos permite comprobar que la fuente contiene diferentes tipos de información:

- **Temporal:** fecha, hora, día, mes y año del siniestro.
- **Descriptiva:** clase y gravedad del siniestro.
- **Ubicación:** dirección, barrio, comuna, latitud y longitud.
- **Geográfica:** cada registro se representa espacialmente como un punto.

La presencia de latitud y longitud es especialmente importante para el proyecto, ya que permitirá plantear posteriormente una relación espacial entre los siniestros registrados y el tramo vial correspondiente a La Asomadera #2.

## 3. Consulta de registros reales

Después de identificar la estructura de la fuente, se consulta una pequeña muestra de registros reales. El objetivo es observar cómo se encuentran almacenados los valores y entender qué representa cada registro antes de seleccionar las variables que serán utilizadas en el proyecto.

In [ ]:
# Endpoint utilizado para consultar los registros de la capa
url_consulta = f"{url_capa}/query"

parametros = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "resultRecordCount": 10,
    "f": "json"
}

respuesta_datos = requests.get(
    url_consulta,
    params=parametros,
    timeout=30
)

respuesta_datos.raise_for_status()

datos = respuesta_datos.json()

# Verificamos que la API no haya devuelto un error
if "error" in datos:
    print("La API devolvió un error:")
    print(datos["error"])
else:
    print(f"{len(datos.get("features", []))} Consultadoscorrectamente")

10 Consultadoscorrectamente


In [ ]:
# Extraemos los atributos de cada registro
registros = [
    feature["attributes"]
    for feature in datos["features"]
]

df_muestra = pd.DataFrame(registros)

df_muestra

,objectid,radicado,fecha_siniestro,hora_siniestro,dia,mes,anio,clase_siniestro,gravedad_siniestro,direccion,barrio,num_comuna,nom_comuna,latitud,longitud
0,1,1655171.0,2019-01-01,00:10:00,martes,enero,2019.0,ATROPELLO,HERIDOS,CL 85 B ENTRE CR 32 34,Las Granjas,Comuna 3,Manrique,6.276538,-75.545609
1,2,1655420.0,2019-01-01,00:20:00,martes,enero,2019.0,CHOQUE,DAÑOS,CR 70 CL 50,Carlos E. Restrepo,Comuna 11,Laureles Estadio,6.258919,-75.585837
2,3,1655135.0,2019-01-01,00:30:00,martes,enero,2019.0,OTRO,HERIDOS,CL 113 POR CR 73,Belén,Comuna 16,Belen,6.233740,-75.594340
3,4,1655189.0,2019-01-01,00:40:00,martes,enero,2019.0,ATROPELLO,HERIDOS,cl 42 cr 94,Campo Alegre,Comuna 12,La América,6.253970,-75.610860
4,5,1655109.0,2019-01-01,00:42:00,martes,enero,2019.0,CHOQUE,HERIDOS,CRA 39 FRENTE 92B 129,El Poblado,Comuna 14,El Poblado,6.208380,-75.568060
5,6,1655118.0,2019-01-01,00:50:00,martes,enero,2019.0,CHOQUE,HERIDOS,CR 45 POR CL 57,Villa Nueva,Comuna 10,La Candelaria,6.253026,-75.560700
6,7,1655143.0,2019-01-01,01:00:00,martes,enero,2019.0,ATROPELLO,HERIDOS,CR 46 FRENTE # 108 42,Popular,Comuna 1,Popular,6.297975,-75.550074
7,8,1655426.0,2019-01-01,01:00:00,martes,enero,2019.0,CHOQUE,DAÑOS,CR 55 ENTRE CLS 92 Y 93,Palermo,Comuna 4,Aranjuez,6.285342,-75.564182
8,9,1655468.0,2019-01-01,01:00:00,martes,enero,2019.0,CHOQUE,DAÑOS,cl 67 ALTURA CR 51 D,Sevilla,Comuna 10,La Candelaria,6.264125,-75.565426
9,10,1655165.0,2019-01-01,01:20:00,martes,enero,2019.0,ATROPELLO,HERIDOS,CL 57A 29 - 73,Enciso,Comuna 8,Villa Hermosa,6.247203,-75.550022


### Información geográfica de los registros

Además de las columnas `latitud` y `longitud`, la API de ArcGIS puede devolver una geometría asociada a cada siniestro.

Se consulta esta geometría porque, más adelante, la ubicación será el elemento que permitirá plantear la relación entre los registros de siniestralidad y el segmento vial correspondiente a La Asomadera #2.

Por ahora solo se verifica que esta información espacial exista; todavía no se realizará ningún cruce entre fuentes.

In [ ]:
# Observamos la geometría asociada al primer registro
primera_geometria = datos["features"][0].get("geometry")

print("Geometría del primer registro:")
print(primera_geometria)

Geometría del primer registro:
{'x': 4718445.705, 'y': 2252229.9276}


In [ ]:
print("Dimensiones de la muestra:", df_muestra.shape)

print("\nTipos de datos interpretados por pandas:")
print(df_muestra.dtypes)

Dimensiones de la muestra: (10, 15)

Tipos de datos interpretados por pandas:
objectid                int64
radicado              float64
fecha_siniestro        object
hora_siniestro         object
dia                    object
mes                    object
anio                  float64
clase_siniestro        object
gravedad_siniestro     object
direccion              object
barrio                 object
num_comuna             object
nom_comuna             object
latitud               float64
longitud              float64
dtype: object


## 4. Adquisición de los datos de la fuente

Después de comprobar el acceso y la estructura de la API, se realiza la extracción de los registros disponibles para conservar una copia de los datos originales.

En esta etapa los datos se almacenan sin realizar limpieza, transformación de tipos ni filtrado espacial hacia La Asomadera #2. De esta forma se conserva una versión de la información obtenida directamente desde la fuente.

In [ ]:
# Extracción de todos los registros disponibles mediante paginación

todos_los_registros = []
offset = 0
tamano_bloque = 2000

while True:
    parametros_extraccion = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "resultOffset": offset,
        "resultRecordCount": tamano_bloque,
        "f": "json"
    }

    respuesta = requests.get(
        url_consulta,
        params=parametros_extraccion,
        timeout=60
    )

    respuesta.raise_for_status()
    datos_pagina = respuesta.json()

    features = datos_pagina.get("features", [])

    if not features:
        break

    todos_los_registros.extend(
        feature["attributes"] for feature in features
    )

    print(
        f"Registros obtenidos hasta el momento: "
        f"{len(todos_los_registros)}"
    )

    if len(features) < tamano_bloque:
        break

    offset += tamano_bloque

df_siniestralidad_raw = pd.DataFrame(todos_los_registros)

print("\nExtracción finalizada")
print("Total de registros:", len(df_siniestralidad_raw))

df_siniestralidad_raw.head()

Registros obtenidos hasta el momento: 2000
Registros obtenidos hasta el momento: 4000
Registros obtenidos hasta el momento: 6000
Registros obtenidos hasta el momento: 8000
Registros obtenidos hasta el momento: 10000
Registros obtenidos hasta el momento: 12000
Registros obtenidos hasta el momento: 14000
Registros obtenidos hasta el momento: 16000
Registros obtenidos hasta el momento: 18000
Registros obtenidos hasta el momento: 20000
Registros obtenidos hasta el momento: 22000
Registros obtenidos hasta el momento: 24000
Registros obtenidos hasta el momento: 26000
Registros obtenidos hasta el momento: 28000
Registros obtenidos hasta el momento: 30000
Registros obtenidos hasta el momento: 32000
Registros obtenidos hasta el momento: 34000
Registros obtenidos hasta el momento: 36000
Registros obtenidos hasta el momento: 38000
Registros obtenidos hasta el momento: 40000
Registros obtenidos hasta el momento: 42000
Registros obtenidos hasta el momento: 44000
Registros obtenidos hasta el momento

,objectid,radicado,fecha_siniestro,hora_siniestro,dia,mes,anio,clase_siniestro,gravedad_siniestro,direccion,barrio,num_comuna,nom_comuna,latitud,longitud
0,1,1655171.0,2019-01-01,00:10:00,martes,enero,2019.0,ATROPELLO,HERIDOS,CL 85 B ENTRE CR 32 34,Las Granjas,Comuna 3,Manrique,6.276538,-75.545609
1,2,1655420.0,2019-01-01,00:20:00,martes,enero,2019.0,CHOQUE,DAÑOS,CR 70 CL 50,Carlos E. Restrepo,Comuna 11,Laureles Estadio,6.258919,-75.585837
2,3,1655135.0,2019-01-01,00:30:00,martes,enero,2019.0,OTRO,HERIDOS,CL 113 POR CR 73,Belén,Comuna 16,Belen,6.233740,-75.594340
3,4,1655189.0,2019-01-01,00:40:00,martes,enero,2019.0,ATROPELLO,HERIDOS,cl 42 cr 94,Campo Alegre,Comuna 12,La América,6.253970,-75.610860
4,5,1655109.0,2019-01-01,00:42:00,martes,enero,2019.0,CHOQUE,HERIDOS,CRA 39 FRENTE 92B 129,El Poblado,Comuna 14,El Poblado,6.208380,-75.568060


### Almacenamiento de la fuente adquirida

Los registros obtenidos se almacenan en un archivo CSV conservando los campos y valores recibidos desde la API. Este archivo funcionará como fuente de entrada para las etapas posteriores del proyecto.

In [ ]:
nombre_archivo = "siniestralidad_medellin.csv"

df_siniestralidad_raw.to_csv(
    nombre_archivo,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Archivo generado:", nombre_archivo)
print("Registros almacenados:", len(df_siniestralidad_raw))

Archivo generado: siniestralidad_medellin.csv
Registros almacenados: 235820


###DESCARGAR

In [ ]:
from google.colab import files

files.download("siniestralidad_medellin.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Calidad de los datos

Se realiza un análisis de calidad sobre todas las variables descargadas, identificando datos faltantes, posibles valores que representan ausencia de información y errores de acuerdo con reglas básicas de consistencia.

Posteriormente se revisan específicamente las variables relevantes para el proyecto.

In [ ]:
import pandas as pd
from IPython.display import display

# =========================================================
# 6. CALIDAD DE LOS DATOS - SINIESTRALIDAD
# =========================================================

# Leemos el archivo generado previamente por el notebook
df = pd.read_csv(
    "siniestralidad_medellin.csv",
    sep=";"
)

print("Registros analizados:", len(df))
print("Variables analizadas:", len(df.columns))


# ---------------------------------------------------------
# 1. Identificación de datos faltantes
# ---------------------------------------------------------

marcadores_sin_informacion = {
    "ND",
    "N/D",
    "SIN INFORMACION",
    "SIN INFORMACIÓN",
    "NO DISPONIBLE",
    "NULL"
}


def analizar_ausencia(serie):

    # NaN / None
    faltante = serie.isna().copy()

    # Valores codificados como ausencia
    codificado = pd.Series(
        False,
        index=serie.index
    )

    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):
        texto = (
            serie.astype("string")
            .str.strip()
        )

        # Cadena vacía
        faltante = (
            faltante
            | texto.eq("").fillna(False)
        )

        # ND, N/D, etc.
        codificado = (
            texto.str.upper()
            .isin(marcadores_sin_informacion)
            .fillna(False)
        )

    return faltante, codificado


faltantes = {}
faltantes_codificados = {}

for columna in df.columns:

    mascara_faltante, mascara_codificada = (
        analizar_ausencia(df[columna])
    )

    faltantes[columna] = mascara_faltante
    faltantes_codificados[columna] = mascara_codificada


# ---------------------------------------------------------
# 2. Máscaras de error
# ---------------------------------------------------------

errores = {
    columna: pd.Series(
        False,
        index=df.index
    )
    for columna in df.columns
}

criterios = {
    columna: "Se revisan datos faltantes"
    for columna in df.columns
}


# ---------------------------------------------------------
# objectid
# ---------------------------------------------------------

objectid = pd.to_numeric(
    df["objectid"],
    errors="coerce"
)

errores["objectid"] = (
    ~faltantes["objectid"]
    & (
        objectid.isna()
        | (objectid <= 0)
        | df["objectid"].duplicated(keep=False)
    )
)

criterios["objectid"] = (
    "Numérico, positivo y no duplicado"
)


# ---------------------------------------------------------
# radicado
# ---------------------------------------------------------

radicado = pd.to_numeric(
    df["radicado"],
    errors="coerce"
)

errores["radicado"] = (
    ~faltantes["radicado"]
    & (
        radicado.isna()
        | (radicado <= 0)
    )
)

criterios["radicado"] = (
    "Numérico y positivo"
)


# ---------------------------------------------------------
# fecha_siniestro
# ---------------------------------------------------------

fecha = pd.to_datetime(
    df["fecha_siniestro"],
    errors="coerce"
)

errores["fecha_siniestro"] = (
    ~faltantes["fecha_siniestro"]
    & (
        fecha.isna()
        | ~fecha.dt.year.between(2019, 2025)
    )
)

criterios["fecha_siniestro"] = (
    "Fecha válida entre 2019 y 2025"
)


# ---------------------------------------------------------
# hora_siniestro
# ---------------------------------------------------------

hora = pd.to_datetime(
    df["hora_siniestro"].astype("string"),
    format="%H:%M:%S",
    errors="coerce"
)

errores["hora_siniestro"] = (
    ~faltantes["hora_siniestro"]
    & hora.isna()
)

criterios["hora_siniestro"] = (
    "Hora válida en formato HH:MM:SS"
)


# ---------------------------------------------------------
# Día, mes y año
# ---------------------------------------------------------

dias_semana = {
    0: "lunes",
    1: "martes",
    2: "miércoles",
    3: "jueves",
    4: "viernes",
    5: "sábado",
    6: "domingo"
}

meses = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"
}


dia_registrado = (
    df["dia"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mes_registrado = (
    df["mes"]
    .astype("string")
    .str.strip()
    .str.lower()
)

anio = pd.to_numeric(
    df["anio"],
    errors="coerce"
)


errores["dia"] = (
    ~faltantes["dia"]
    & fecha.notna()
    & (
        dia_registrado
        != fecha.dt.dayofweek.map(dias_semana)
    )
)

criterios["dia"] = (
    "Debe coincidir con fecha_siniestro"
)


errores["mes"] = (
    ~faltantes["mes"]
    & fecha.notna()
    & (
        mes_registrado
        != fecha.dt.month.map(meses)
    )
)

criterios["mes"] = (
    "Debe coincidir con fecha_siniestro"
)


errores["anio"] = (
    ~faltantes["anio"]
    & (
        anio.isna()
        | ~anio.between(2019, 2025)
        | (
            fecha.notna()
            & (anio != fecha.dt.year)
        )
    )
)

criterios["anio"] = (
    "Debe coincidir con la fecha y estar entre 2019 y 2025"
)


# ---------------------------------------------------------
# Latitud
# ---------------------------------------------------------

latitud = pd.to_numeric(
    df["latitud"],
    errors="coerce"
)

errores["latitud"] = (
    ~faltantes["latitud"]
    & (
        latitud.isna()
        | ~latitud.between(-90, 90)
        | (latitud == 0)
    )
)

criterios["latitud"] = (
    "Numérica entre -90 y 90 y diferente de 0"
)


# ---------------------------------------------------------
# Longitud
# ---------------------------------------------------------

longitud = pd.to_numeric(
    df["longitud"],
    errors="coerce"
)

errores["longitud"] = (
    ~faltantes["longitud"]
    & (
        longitud.isna()
        | ~longitud.between(-180, 180)
        | (longitud == 0)
    )
)

criterios["longitud"] = (
    "Numérica entre -180 y 180 y diferente de 0"
)


# =========================================================
# 3. TABLA 1 - TODAS LAS VARIABLES
# =========================================================

tabla_calidad_completa = pd.DataFrame({
    "Variable": df.columns,
    "Tipo de dato": df.dtypes.astype(str).values,
    "Datos faltantes": [
        int(faltantes[col].sum())
        for col in df.columns
    ],
    "Faltantes codificados": [
        int(faltantes_codificados[col].sum())
        for col in df.columns
    ],
    "Datos con error": [
        int(errores[col].sum())
        for col in df.columns
    ],
    "Criterio revisado": [
        criterios[col]
        for col in df.columns
    ]
})


print("\nTABLA 1 - CALIDAD DE TODAS LAS VARIABLES")
display(tabla_calidad_completa)


# =========================================================
# 4. TABLA 2 - VARIABLES RELEVANTES PARA EL PROYECTO
# =========================================================

variables_relevantes = [
    "fecha_siniestro",
    "hora_siniestro",
    "clase_siniestro",
    "gravedad_siniestro",
    "latitud",
    "longitud"
]

tabla_calidad_relevantes = (
    tabla_calidad_completa[
        tabla_calidad_completa["Variable"]
        .isin(variables_relevantes)
    ]
    .reset_index(drop=True)
)


print("\nTABLA 2 - VARIABLES RELEVANTES PARA EL PROYECTO")
display(tabla_calidad_relevantes)


# =========================================================
# 5. RESUMEN
# =========================================================

ausencias_completas = pd.DataFrame({
    col: (
        faltantes[col]
        | faltantes_codificados[col]
    )
    for col in df.columns
})

errores_completos = pd.DataFrame(errores)


ausencias_relevantes = pd.DataFrame({
    col: (
        faltantes[col]
        | faltantes_codificados[col]
    )
    for col in variables_relevantes
})

errores_relevantes = pd.DataFrame({
    col: errores[col]
    for col in variables_relevantes
})


print("\nRESUMEN GENERAL")
print(
    "Registros con alguna ausencia de información:",
    int(ausencias_completas.any(axis=1).sum())
)

print(
    "Registros con algún error:",
    int(errores_completos.any(axis=1).sum())
)


print("\nRESUMEN VARIABLES RELEVANTES")
print(
    "Registros con alguna ausencia:",
    int(ausencias_relevantes.any(axis=1).sum())
)

print(
    "Registros con algún error:",
    int(errores_relevantes.any(axis=1).sum())
)

Registros analizados: 235820
Variables analizadas: 15

TABLA 1 - CALIDAD DE TODAS LAS VARIABLES


,Variable,Tipo de dato,Datos faltantes,Faltantes codificados,Datos con error,Criterio revisado
0,objectid,int64,0,0,0,"Numérico, positivo y no duplicado"
1,radicado,float64,0,0,0,Numérico y positivo
2,fecha_siniestro,object,0,0,0,Fecha válida entre 2019 y 2025
3,hora_siniestro,object,0,0,0,Hora válida en formato HH:MM:SS
4,dia,object,0,0,0,Debe coincidir con fecha_siniestro
5,mes,object,0,0,0,Debe coincidir con fecha_siniestro
6,anio,float64,0,0,0,Debe coincidir con la fecha y estar entre 2019...
7,clase_siniestro,object,0,0,0,Se revisan datos faltantes
8,gravedad_siniestro,object,0,0,0,Se revisan datos faltantes
9,direccion,object,0,0,0,Se revisan datos faltantes



TABLA 2 - VARIABLES RELEVANTES PARA EL PROYECTO


,Variable,Tipo de dato,Datos faltantes,Faltantes codificados,Datos con error,Criterio revisado
0,fecha_siniestro,object,0,0,0,Fecha válida entre 2019 y 2025
1,hora_siniestro,object,0,0,0,Hora válida en formato HH:MM:SS
2,clase_siniestro,object,0,0,0,Se revisan datos faltantes
3,gravedad_siniestro,object,0,0,0,Se revisan datos faltantes
4,latitud,float64,0,0,0,Numérica entre -90 y 90 y diferente de 0
5,longitud,float64,0,0,0,Numérica entre -180 y 180 y diferente de 0



RESUMEN GENERAL
Registros con alguna ausencia de información: 473
Registros con algún error: 0

RESUMEN VARIABLES RELEVANTES
Registros con alguna ausencia: 0
Registros con algún error: 0
